# PySpark: Gold Layer Consolidado
Pipeline PySpark que une os 3 datasets da camada Prata (versão ML-Ready, sem data leakage) e gera a camada Ouro.

**Pré-requisito:** Executar TODAS as células (incluindo a seção PySpark) dos 3 notebooks individuais
(`incedets_master.ipynb`, `financial_impact.ipynb`, `market_impact.ipynb`) para gerar os
arquivos Silver Spark (`*_ml_spark.parquet`).

### Etapas:
1. **Configuração**: SparkSession + helper `salvar_parquet` para gravação sem subpastas
2. **Window Functions**: enriquece cada Silver ML com métricas analíticas
3. **Gold Layer**: join dos 3 datasets ML-Ready + agregações + rankings
4. **Flag de nulos imputados**: marca registros com valores originalmente nulos para exclusão opcional no treino
5. **Comparação de desempenho** Pandas vs PySpark
6. **Tabela de equivalência** de operações


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    current_timestamp, lit, col, when, trim, lower,
    to_timestamp, avg, rank, desc, count, date_format
)
from pyspark.sql.window import Window
import time, numpy as np, pandas as pd, subprocess, sys, shutil
from pathlib import Path

# ── Verifica se Java está instalado ───────────────────────────────
java_ok = False
try:
    subprocess.run(["java", "-version"], capture_output=True, timeout=5)
    java_ok = True
except Exception:
    pass

if not java_ok:
    print("=" * 60)
    print("ERRO: Java nao encontrado! PySpark exige Java JDK 11+.")
    print("Instale o Java (https://adoptium.net) e configure JAVA_HOME.")
    print("=" * 60)
    spark = None
else:
    spark = (
        SparkSession.builder
        .appName("GoldLayerPySpark")
        .config("spark.sql.adaptive.enabled", "true")
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    print("SparkSession OK")


def salvar_parquet(df, caminho: str) -> None:
    """
    Salva um DataFrame Spark como arquivo Parquet ÚNICO no caminho especificado,
    sem criar subpastas intermediárias (comportamento padrão do Spark).

    Estratégia:
      1. coalesce(1)  → consolida em 1 partição para gerar um único arquivo
      2. Grava em diretório temporário _tmp_<nome>
      3. Localiza o único arquivo part-*.parquet gerado
      4. Move-o para o caminho definitivo (substitui se existir)
      5. Remove o diretório temporário

    Parâmetros
    ----------
    df      : DataFrame PySpark
    caminho : caminho final do arquivo, ex. 'Dados/ouro/gold_completo.parquet'
    """
    caminho = Path(caminho)
    caminho.parent.mkdir(parents=True, exist_ok=True)

    tmp_dir = caminho.parent / f"_tmp_{caminho.stem}"

    # Garante que o diretório temporário está limpo
    if tmp_dir.exists():
        shutil.rmtree(str(tmp_dir))

    # Grava como partição única
    df.coalesce(1).write.mode("overwrite").parquet(str(tmp_dir))

    # Localiza o arquivo gerado pelo Spark
    parquet_files = list(tmp_dir.glob("part-*.parquet"))
    if not parquet_files:
        raise FileNotFoundError(
            f"Nenhum arquivo part-*.parquet encontrado em {tmp_dir}"
        )

    # Remove destino anterior se existir
    if caminho.exists():
        caminho.unlink()

    shutil.move(str(parquet_files[0]), str(caminho))
    shutil.rmtree(str(tmp_dir))
    print(f"  Salvo: {caminho}")


---
## 1. Window Functions
Operações analíticas nativas do Spark aplicadas sobre cada dataset Silver ML-Ready.

> **Nota:** A partir desta versão, todas as etapas utilizam os arquivos `*_ml_spark.parquet`
> (versão sem colunas de data leakage), garantindo que o dataset final da camada Ouro
> esteja seguro para uso em Machine Learning.


### 1.1 Window — incedets_master
Média móvel acumulada de registros comprometidos por vetor de ataque + ranking por setor.


In [ ]:
if spark:
    t0 = time.time()

    # Lê versão ML-Ready (sem colunas de data leakage)
    df_m = spark.read.parquet("Dados/prata/incedets_master_silver_ml_spark.parquet")

    # Window 1: média móvel acumulada de registros comprometidos por vetor de ataque
    w_ataque = (
        Window.partitionBy("attack_vector_primary")
              .orderBy("incident_date")
              .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    # Window 2: ranking de registros comprometidos dentro de cada setor
    w_setor = (
        Window.partitionBy("industry_primary")
              .orderBy(desc("data_compromised_records"))
    )

    df_w_m = (
        df_m
        .withColumn(
            "avg_records_por_ataque",
            avg("data_compromised_records").over(w_ataque)
        )
        .withColumn(
            "rank_records_setor",
            rank().over(w_setor)
        )
    )

    salvar_parquet(df_w_m, "Dados/ouro/window_incedets_master.parquet")
    t_window = time.time() - t0
    print(f"Window incedets_master: {t_window:.4f}s | Colunas: {len(df_w_m.columns)}")
else:
    print("Spark nao disponivel. Pule esta celula.")


### 1.2 Window — financial_impact
Média móvel acumulada de perda direta por método de cálculo + ranking por índice CPI.


In [ ]:
if spark:
    t0 = time.time()

    # Lê versão ML-Ready (sem colunas de data leakage)
    df_f = spark.read.parquet("Dados/prata/financial_impact_prata_ml_spark.parquet")

    # Window 1: média móvel acumulada de perda direta por método de cálculo
    w_metodo = (
        Window.partitionBy("direct_loss_method")
              .orderBy("created_at")
              .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    # Window 2: ranking de perda direta dentro de cada índice CPI
    w_cpi = (
        Window.partitionBy("cpi_index_used")
              .orderBy(desc("direct_loss_usd"))
    )

    df_w_f = (
        df_f
        .withColumn(
            "avg_direct_loss_por_metodo",
            avg("direct_loss_usd").over(w_metodo)
        )
        .withColumn(
            "rank_perda_por_cpi",
            rank().over(w_cpi)
        )
    )

    salvar_parquet(df_w_f, "Dados/ouro/window_financial_impact.parquet")
    t_window = time.time() - t0
    print(f"Window financial_impact: {t_window:.4f}s | Colunas: {len(df_w_f.columns)}")
else:
    print("Spark nao disponivel. Pule esta celula.")


### 1.3 Window — market_impact
Média móvel acumulada de market cap por setor + ranking de volume por ticker.


In [ ]:
if spark:
    t0 = time.time()

    # Lê versão ML-Ready (sem colunas de data leakage)
    df_k = spark.read.parquet("Dados/prata/market_silver_ml_spark.parquet")

    # Window 1: média móvel acumulada de market cap por setor
    w_setor = (
        Window.partitionBy("sector_index")
              .orderBy("created_at")
              .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )
    # Window 2: ranking de volume de negociação dentro de cada ticker
    w_ticker = (
        Window.partitionBy("stock_ticker")
              .orderBy(desc("volume_ratio_disclosure"))
    )

    df_w_k = (
        df_k
        .withColumn(
            "avg_marketcap_por_setor",
            avg("market_cap_at_disclosure").over(w_setor)
        )
        .withColumn(
            "rank_volume_ticker",
            rank().over(w_ticker)
        )
    )

    salvar_parquet(df_w_k, "Dados/ouro/window_market_impact.parquet")
    t_window = time.time() - t0
    print(f"Window market_impact: {t_window:.4f}s | Colunas: {len(df_w_k.columns)}")
else:
    print("Spark nao disponivel. Pule esta celula.")


---
## 2. Gold Layer — Join + Agregações + Rankings
Junção dos 3 datasets Silver ML-Ready via `incident_id` e geração da camada Ouro.

**Fontes (todas sem data leakage):**
- `incedets_master_silver_ml_spark.parquet`
- `financial_impact_prata_ml_spark.parquet`
- `market_silver_ml_spark.parquet`


In [ ]:
if spark:
    t0 = time.time()

    # ── Leitura dos datasets ML-Ready (sem data leakage) ──────────
    m = spark.read.parquet("Dados/prata/incedets_master_silver_ml_spark.parquet")
    f = spark.read.parquet("Dados/prata/financial_impact_prata_ml_spark.parquet")
    k = spark.read.parquet("Dados/prata/market_silver_ml_spark.parquet")

    # ── Join 1: incidents + financial (left join por incident_id) ──
    dup_mf = [c for c in f.columns if c != "incident_id" and c in m.columns]
    f_clean = f.drop(*dup_mf)
    gold = m.join(f_clean, on="incident_id", how="left")

    # ── Join 2: resultado + market (left join por incident_id) ─────
    dup_mk = [c for c in k.columns if c != "incident_id" and c in gold.columns]
    k_clean = k.drop(*dup_mk)
    gold = gold.join(k_clean, on="incident_id", how="left")

    print(f"Gold: {gold.count()} registros | {len(gold.columns)} colunas")

    # ── groupBy + agg: métricas médias por setor ──────────────────
    # Nota: total_loss_usd foi removido por leakage; usa direct_loss_usd como proxy
    agg_setor = gold.groupBy("industry_primary").agg(
        avg("data_compromised_records").alias("avg_records_comprometidos"),
        avg("direct_loss_usd").alias("avg_direct_loss"),
        avg("market_cap_at_disclosure").alias("avg_marketcap"),
        count("*").alias("total_incidentes")
    )

    # ── Window sobre agregação: ranking setores por perda direta média
    agg_setor = agg_setor.withColumn(
        "rank_setor_prejuizo",
        rank().over(Window.orderBy(desc("avg_direct_loss")))
    )

    agg_setor.show(10, truncate=False)

    # ── Salva como arquivos únicos (sem subpastas) ─────────────────
    salvar_parquet(gold, "Dados/ouro/gold_completo.parquet")
    salvar_parquet(agg_setor, "Dados/ouro/gold_agregacoes_setor.parquet")

    t_gold = time.time() - t0
    print(f"Gold criada em {t_gold:.4f}s")
else:
    print("Spark nao disponivel. Pule esta celula.")


---
## 3. Flag de Nulos Imputados — Dataset ML-Ready Final
Durante a camada Prata, valores nulos foram preenchidos com `0` (numéricos) ou `'desconhecido'` (categóricos).
Para Machine Learning, registros com valores originalmente ausentes podem introduzir ruído no modelo.

Esta etapa:
1. **Detecta** colunas numéricas-chave com valor `0` proveniente de imputação
2. **Cria** a coluna `_flag_nulos_imputados` (1 = ao menos uma coluna foi imputada; 0 = dados completos originais)
3. **Salva** o `dataset_ml_ouro.parquet` contendo apenas registros nativamente completos

> O time de ML pode escolher entre treinar com todos os dados ou apenas com registros sem imputação.


In [ ]:
if spark:
    t0 = time.time()

    gold_ml = spark.read.parquet("Dados/ouro/gold_completo.parquet")

    # ── Colunas imputadas com 0 na camada Prata ────────────────────
    # Registros com valor 0 nessas colunas PODEM ter sido nulos originalmente.
    cols_imputadas = [
        "company_revenue_usd",       # incedets_master: nulos → 0
        "employee_count",            # incedets_master: nulos → 0
        "data_compromised_records",  # incedets_master: nulos → 0
        "direct_loss_usd",           # financial_impact: nulos → 0
        "ransom_demanded_usd",       # financial_impact: nulos → 0
        "market_cap_at_disclosure",  # market_impact: nulos → 0
        "volume_ratio_disclosure",   # market_impact: nulos → 0
        "pre_incident_volatility_30d",  # market_impact: nulos → 0
    ]

    # Filtra apenas as colunas presentes no dataset Gold
    cols_existentes = [c for c in cols_imputadas if c in gold_ml.columns]

    # ── Cria flag: 1 se QUALQUER coluna imputada tiver valor 0 ─────
    flag_expr = when(col(cols_existentes[0]) == 0, 1).otherwise(0)
    for c in cols_existentes[1:]:
        flag_expr = when((flag_expr == 1) | (col(c) == 0), 1).otherwise(0)

    gold_ml = gold_ml.withColumn("_flag_nulos_imputados", flag_expr)

    total    = gold_ml.count()
    com_flag = gold_ml.filter(col("_flag_nulos_imputados") == 1).count()
    sem_flag = total - com_flag
    print(f"Total de registros          : {total}")
    print(f"Com nulos imputados (_flag=1): {com_flag} ({com_flag / total * 100:.1f}%)")
    print(f"Nativamente completos (_flag=0): {sem_flag} ({sem_flag / total * 100:.1f}%)")

    # ── Salva Gold completo com flag (substitui versão anterior) ───
    salvar_parquet(gold_ml, "Dados/ouro/gold_completo.parquet")

    # ── Salva dataset ML-Ready: apenas registros sem imputacao ─────
    gold_sem_imputacao = gold_ml.filter(col("_flag_nulos_imputados") == 0)
    salvar_parquet(
        gold_sem_imputacao.drop("_flag_nulos_imputados"),
        "Dados/ouro/dataset_ml_ouro.parquet"
    )
    print(f"dataset_ml_ouro.parquet: {gold_sem_imputacao.count()} registros sem imputacao")

    t_flag = time.time() - t0
    print(f"Flag de nulos criada em {t_flag:.4f}s")
else:
    print("Spark nao disponivel. Pule esta celula.")


---
## 4. Comparação Pandas vs PySpark
Benchmark de desempenho entre as duas implementações para cada dataset.


### 4.1 incedets_master


In [ ]:
if spark:
    cols_num = ["company_revenue_usd", "employee_count",
                "data_compromised_records", "downtime_hours"]

    # ── PANDAS ────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv("Dados/originais/incidents_master.csv")
    df_pd = df_pd.drop_duplicates(subset=["incident_id"], keep="last")
    for c in cols_num:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].fillna(0)
    t_pd = time.time() - t0

    # ── PySpark ───────────────────────────────────────────────────
    t0 = time.time()
    df_sp = spark.read.parquet("Dados/bronze/incedets_master_bronze_spark.parquet")
    df_sp = df_sp.dropDuplicates(["incident_id"])
    for c in cols_num:
        df_sp = df_sp.fillna(0, subset=[c])
    _ = df_sp.count()  # força execucao do plano lazy
    t_sp = time.time() - t0

    ratio = t_pd / t_sp if t_sp > 0 else float("inf")
    print("=" * 60)
    print("COMPARACAO incedets_master: PANDAS vs PySpark")
    print("=" * 60)
    print(f"  Pandas:  {t_pd:.4f}s")
    print(f"  PySpark: {t_sp:.4f}s")
    print(f"  Speedup: {ratio:.2f}x")
    print("=" * 60)
else:
    print("Spark nao disponivel. Pule esta celula.")


### 4.2 financial_impact


In [ ]:
if spark:
    cols_num = [
        "direct_loss_usd", "ransom_demanded_usd", "ransom_paid_usd",
        "recovery_cost_usd", "legal_fees_usd", "regulatory_fine_usd",
        "insurance_payout_usd", "total_loss_usd", "total_loss_lower_bound",
        "total_loss_upper_bound", "inflation_adjusted_usd"
    ]

    # ── PANDAS ────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv("Dados/originais/financial_impact.csv")
    df_pd = df_pd.drop_duplicates(subset=["incident_id"], keep="last")
    for c in cols_num:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].fillna(0)
    t_pd = time.time() - t0

    # ── PySpark ───────────────────────────────────────────────────
    t0 = time.time()
    df_sp = spark.read.parquet("Dados/bronze/financial_impact_bronze_spark.parquet")
    df_sp = df_sp.dropDuplicates(["incident_id"])
    for c in cols_num:
        df_sp = df_sp.fillna(0, subset=[c])
    _ = df_sp.count()
    t_sp = time.time() - t0

    ratio = t_pd / t_sp if t_sp > 0 else float("inf")
    print("=" * 60)
    print("COMPARACAO financial_impact: PANDAS vs PySpark")
    print("=" * 60)
    print(f"  Pandas:  {t_pd:.4f}s")
    print(f"  PySpark: {t_sp:.4f}s")
    print(f"  Speedup: {ratio:.2f}x")
    print("=" * 60)
else:
    print("Spark nao disponivel. Pule esta celula.")


### 4.3 market_impact


In [ ]:
if spark:
    cols_num = [
        "market_cap_at_disclosure", "volume_ratio_disclosure",
        "pre_incident_volatility_30d", "post_incident_volatility_30d",
        "days_to_price_recovery", "car_0_to_30", "abnormal_return_30d",
        "t_statistic_30d", "p_value_30d"
    ]

    # ── PANDAS ────────────────────────────────────────────────────
    t0 = time.time()
    df_pd = pd.read_csv("Dados/originais/market_impact.csv")
    df_pd = df_pd.drop_duplicates(subset=["incident_id"], keep="last")
    for c in cols_num:
        if c in df_pd.columns:
            df_pd[c] = df_pd[c].fillna(0)
    t_pd = time.time() - t0

    # ── PySpark ───────────────────────────────────────────────────
    t0 = time.time()
    df_sp = spark.read.parquet("Dados/bronze/market_impact_bronze_spark.parquet")
    df_sp = df_sp.dropDuplicates(["incident_id"])
    for c in cols_num:
        df_sp = df_sp.fillna(0, subset=[c])
    _ = df_sp.count()
    t_sp = time.time() - t0

    ratio = t_pd / t_sp if t_sp > 0 else float("inf")
    print("=" * 60)
    print("COMPARACAO market_impact: PANDAS vs PySpark")
    print("=" * 60)
    print(f"  Pandas:  {t_pd:.4f}s")
    print(f"  PySpark: {t_sp:.4f}s")
    print(f"  Speedup: {ratio:.2f}x")
    print("=" * 60)
else:
    print("Spark nao disponivel. Pule esta celula.")


---
## 5. Tabela de Equivalência Pandas → PySpark

| Operação | Pandas | PySpark |
|---|---|---|
| Leitura CSV | `pd.read_csv()` | `spark.read.csv()` |
| Auditoria timestamp | `df['col'] = dt.now()` | `.withColumn(..., current_timestamp())` |
| Deduplicação | `df.drop_duplicates()` | `df.dropDuplicates()` |
| Fill nulos | `df[col].fillna(0)` | `df.fillna(0, subset=[col])` |
| Normalização string | `str.lower().strip()` | `lower(trim(col()))` |
| Parse de datas | `pd.to_datetime()` | `to_timestamp(col())` |
| Label binário | `np.where(cond, 1, 0)` | `when(cond, 1).otherwise(0)` |
| Anti-leakage | `df.drop(columns=...)` | `df.drop(*cols)` |
| Join | `pd.merge(df1, df2, on=key)` | `df1.join(df2, on=key, how=...)` |
| Agregação | `df.groupby('c').agg(...)` | `df.groupBy('c').agg(...)` |
| Window functions | Não nativa | `Window.partitionBy().orderBy()` |
| **Salvar sem subpastas** | `df.to_parquet(path)` | `coalesce(1)` + move `part-*.parquet` |
| Flag de imputados | `df[mask].assign(flag=1)` | `when(col==0, 1).otherwise(0)` |


---
### Conclusão

A camada Ouro foi gerada com sucesso a partir dos datasets ML-Ready (sem data leakage):

| Arquivo | Descrição |
|---|---|
| `Dados/ouro/gold_completo.parquet` | Join completo dos 3 datasets ML + `_flag_nulos_imputados` |
| `Dados/ouro/dataset_ml_ouro.parquet` | Apenas registros nativamente completos — **recomendado para treino ML** |
| `Dados/ouro/gold_agregacoes_setor.parquet` | Agregações por setor com ranking de prejuízo |
| `Dados/ouro/window_incedets_master.parquet` | Window: média móvel de records + ranking por setor |
| `Dados/ouro/window_financial_impact.parquet` | Window: média móvel de perda direta + ranking CPI |
| `Dados/ouro/window_market_impact.parquet` | Window: média móvel de market cap + ranking de volume |

**Melhorias aplicadas nesta versão:**
- `salvar_parquet()`: salva arquivo `.parquet` único sem criar subpastas (resolve comportamento nativo do Spark)
- Todas as etapas Gold consomem os arquivos `*_ml_spark.parquet` (sem colunas de data leakage)
- `_flag_nulos_imputados`: marca registros com valores zerados por imputação na Prata
- `dataset_ml_ouro.parquet`: dataset final para ML com apenas registros originalmente completos
- Window Functions adaptadas às colunas disponíveis nos datasets ML-Ready
